In [2]:
# Importing libraries
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [4]:
# Connect to the database
conn = sqlite3.connect('../data/credit_union_data.db')
# Test the connection
test_query = "SELECT COUNT(*) as total_members FROM members;"
result = pd.read_sql_query(test_query, conn)

print(f"Connection successful! Total members: {result['total_members'][0]}")

Connection successful! Total members: 10000


In [5]:
# PHASE 1: DATA FAMILIARIZATION - Data can be viewed at data/csv_data_preview

In [6]:
# PHASE 2: BASIC QUERY - Total members = 10,000, lets find total members who are active and total who have churned. 

churn_breakdown = pd.read_sql_query("""
SELECT 
  SUM(CASE WHEN churned = 0 THEN 1 ELSE 0 END) as Active_Members,
  SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) as Churned_Members
FROM members
""", conn)

print(churn_breakdown)


   Active_Members  Churned_Members
0            7820             2180


In [ ]:
# Now lets look at what average balances are kept between the two groups 

Averages = pd.read_sql_query("""
    SELECT 
        CASE WHEN churned = 1 THEN 'Churned' ELSE 'Active' END as Status, 
        ROUND(AVG(a.balance),2) as Avg_Balance, 
        COUNT(DISTINCT m.member_id) Total_Members 
    FROM members m 
    JOIN accounts a ON m.member_id = a.member_id 
    GROUP BY Status""", conn)

Averages



,Status,Avg_Balance,Total_Members
0,Active,12361.23,7820
1,Churned,10317.06,2180


In [8]:
# Im interested in transactional behavior. What is the average number of transactions each group performed from 2022 to the end of 2023?
Avg_Transactions = pd.read_sql_query( """WITH member_transaction_counts AS (
    SELECT 
        m.member_id,
        m.churned,
        COUNT(t.transaction_id) as transaction_count
    FROM members m
    JOIN transactions t ON m.member_id = t.member_id
    GROUP BY m.member_id, m.churned
)

SELECT 
    CASE 
        WHEN churned = 1 THEN 'Churned' 
        ELSE 'Active' 
    END as Status,
    COUNT(member_id) as Total_Members,
    ROUND(AVG(transaction_count), 2) as Avg_Transactions_Per_Member
FROM member_transaction_counts
GROUP BY Status; """, conn)
Avg_Transactions

,Status,Total_Members,Avg_Transactions_Per_Member
0,Active,7820,368.86
1,Churned,2180,141.81


In [ ]:
# Lets understand the personas. Which personas have higher churn rates? 
# Note: First Dashboard deliverable

persona_churn = pd.read_sql_query("""SELECT 
persona,
COUNT(*) as Total_Members,
SUM(CASE WHEN churned = 0 THEN 1 ELSE 0 END) as Active_Count,
SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) as Churned_Count,
ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) *100.00 / COUNT(*),2) as Churn_Percentage

FROM members
GROUP BY persona
ORDER BY Churn_Percentage DESC """, conn)
persona_churn

,persona,Total_Members,Active_Count,Churned_Count,Churn_Percentage
0,Loan-Only,1500,843,657,43.80
1,Seasonal Worker,1000,656,344,34.40
2,Emergency User,1000,714,286,28.60
3,Rate Shopper,1500,1125,375,25.00
4,Slow Adopter,1200,1004,196,16.33
5,Digital-First,1800,1545,255,14.17
6,Primary Banker,2000,1933,67,3.35


In [24]:
# Lets see the average number of transactions per persona

persona_transactions = pd.read_sql_query("""WITH member_txn_summary AS (
    SELECT 
        m.member_id,
        m.persona,
        m.churned,
        COUNT(t.transaction_id) as transaction_count
    FROM members m
    LEFT JOIN transactions t ON m.member_id = t.member_id
    GROUP BY m.member_id, m.persona, m.churned
)

SELECT 
    persona,
    COUNT(*) as Total_Members,
    SUM(CASE WHEN churned = 0 THEN 1 ELSE 0 END) as Active_Count,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) as Churned_Count,
    ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as Churn_Percentage,
    ROUND(AVG(transaction_count), 2) as Avg_Transactions_Per_Member
FROM member_txn_summary
GROUP BY persona
ORDER BY Churn_Percentage DESC; """, conn)
persona_transactions


,persona,Total_Members,Active_Count,Churned_Count,Churn_Percentage,Avg_Transactions_Per_Member
0,Loan-Only,1500,843,657,43.80,56.93
1,Seasonal Worker,1000,656,344,34.40,244.51
2,Emergency User,1000,714,286,28.60,313.20
3,Rate Shopper,1500,1125,375,25.00,104.23
4,Slow Adopter,1200,1004,196,16.33,205.60
5,Digital-First,1800,1545,255,14.17,472.37
6,Primary Banker,2000,1933,67,3.35,648.59
